# MMD Embedder — Simple Example

This notebook walks through the [`MMDEmbedder`](../../src/data_meta_map/mmd_embedder.py) interface end-to-end.

1. Verify that the empirical MMD² estimators behave correctly on synthetic Gaussians.
2. Sweep the RBF bandwidth and visualize the median-heuristic optimum.
3. Compare image datasets (`mnist`, `cifar10`, `kmnist`, `letters`) using three different *auxiliary representation backends* — raw pixels, a frozen pretrained ResNet encoder, and a small per-dataset autoencoder — and observe how the choice reshapes the resulting dataset-similarity geometry.
4. Inspect the Random Fourier Features mode: every dataset is mapped to a single fixed-dimensional vector μ̂_P and dataset distances reduce to plain Euclidean norms.

All cells are CPU-friendly. Image-dataset cells download standard `torchvision` benchmarks into `../../data` and may take a few minutes on the first run.

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
import seaborn as sns

from data_meta_map import MMDEmbedder, mmd
from data_meta_map.mmd import (
    RBFKernel,
    median_heuristic,
    mmd2_biased,
    mmd2_unbiased,
)
from data_meta_map.mmd.encoders import (
    GenerativeEncoder,
    PretrainedEncoder,
    RawEncoder,
)

torch.manual_seed(0)
np.random.seed(0)

## 1. Synthetic Gaussians — sanity check

Empirical MMD² should be close to zero when the two samples come from the *same* distribution and grow as the distributions drift apart. We verify both the biased (eq. 3.30 of MMD.pdf) and unbiased (eq. 3.32) estimators.

In [ ]:
n, d = 500, 8
X = torch.randn(n, d)
Y_same = torch.randn(n, d)

shifts = np.linspace(0.0, 3.0, 10)
biased = []
unbiased = []
for s in shifts:
    Y = torch.randn(n, d) + s
    biased.append(float(mmd2_biased(X, Y, kernel='rbf', sigma=1.0)))
    unbiased.append(float(mmd2_unbiased(X, Y, kernel='rbf', sigma=1.0)))

plt.figure(figsize=(6, 4))
plt.plot(shifts, biased, marker='o', label='biased (eq. 3.30)')
plt.plot(shifts, unbiased, marker='x', label='unbiased (eq. 3.32)')
plt.axhline(0.0, color='k', lw=0.5, ls='--')
plt.xlabel('mean shift of Q')
plt.ylabel(r'$\widehat{\mathrm{MMD}}^2(P, Q)$')
plt.title('MMD² vs. mean-shift between two Gaussians')
plt.legend()
plt.tight_layout()
plt.show()

print(f'Same-distribution biased MMD²:   {float(mmd2_biased(X, Y_same)):.4f}')
print(f'Same-distribution unbiased MMD²: {float(mmd2_unbiased(X, Y_same)):.4f}')

## 2. Bandwidth sweep and the median heuristic

The Gaussian-RBF MMD is sensitive to the bandwidth σ. The median heuristic
$\sigma^2 = \mathrm{median}\,\{\|x_i - x_j\|^2\}$
(MMD.pdf p. 57) usually lands close to the value that maximizes test power.

In [ ]:
X = torch.randn(300, 4)
Y = torch.randn(300, 4) + 1.0

sigmas = np.logspace(-1.5, 1.5, 30)
vals = [float(mmd2_biased(X, Y, kernel='rbf', sigma=float(s))) for s in sigmas]
median_sigma = float(median_heuristic(torch.cat([X, Y], dim=0)))

plt.figure(figsize=(6, 4))
plt.semilogx(sigmas, vals, marker='o')
plt.axvline(median_sigma, color='red', ls='--', label=f'median heuristic σ={median_sigma:.2f}')
plt.xlabel('RBF bandwidth σ')
plt.ylabel(r'$\widehat{\mathrm{MMD}}^2(P, Q)$')
plt.title('Bandwidth sensitivity of the MMD estimator')
plt.legend()
plt.tight_layout()
plt.show()

## 3. Dataset comparison with three representation backends

We compare four image datasets via [`MMDEmbedder`](../../src/data_meta_map/mmd_embedder.py).
The `encoder=` argument selects the *auxiliary model* used to map raw inputs into a feature space before MMD is applied:

| Backend | Description |
| --- | --- |
| [`RawEncoder`](../../src/data_meta_map/mmd/encoders.py) | identity — MMD on raw flattened pixels |
| [`PretrainedEncoder`](../../src/data_meta_map/mmd/encoders.py) | frozen ImageNet ResNet-18, all but the final FC layer |
| [`GenerativeEncoder`](../../src/data_meta_map/mmd/encoders.py) | small per-dataset MLP autoencoder — MMD over latent codes |

The corresponding pairwise MMD² matrices are visualized as clustermaps.

In [ ]:
from data_meta_map import datasets as dm_datasets

DATA_ROOT = '../../data'
DATASET_NAMES = ('mnist', 'cifar10', 'kmnist', 'letters')
raw_datasets = [dm_datasets.__dict__[name](root=DATA_ROOT)[0] for name in DATASET_NAMES]

In [ ]:
def pairwise_matrix(encoder, mode_label, max_samples=300):
    embedder = MMDEmbedder(
        mode='distance',
        kernel='rbf',
        bandwidth='median',
        encoder=encoder,
        emb_dim=2,
        max_samples=max_samples,
        device='cpu',
        seed=0,
    )
    D = embedder.compute_pairwise_distances(raw_datasets).cpu().numpy()
    print(f'[{mode_label}] dataset × dataset MMD²:')
    print(np.round(D, 3))
    return D

raw_D = pairwise_matrix(RawEncoder(), 'raw pixels')

In [ ]:
pretrained_encoder = PretrainedEncoder(
    model_name='resnet18',
    pretrained=True,
    image_shape=(3, 224, 224),
    batch_size=32,
    device='cpu',
)
pre_D = pairwise_matrix(pretrained_encoder, 'pretrained ResNet-18')

In [ ]:
ae_encoder = GenerativeEncoder(
    latent_dim=32,
    hidden_dim=128,
    epochs=3,
    batch_size=128,
    seed=0,
    device='cpu',
)
gen_D = pairwise_matrix(ae_encoder, 'AE latent')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
for ax, D, title in zip(
    axes,
    [raw_D, pre_D, gen_D],
    ['Raw pixels', 'Pretrained ResNet-18', 'AE latent'],
):
    sns.heatmap(
        D,
        ax=ax,
        annot=True,
        fmt='.3f',
        xticklabels=DATASET_NAMES,
        yticklabels=DATASET_NAMES,
        cbar=False,
        cmap='viridis_r',
    )
    ax.set_title(title)
fig.suptitle('Pairwise MMD² across auxiliary representations', y=1.02)
plt.tight_layout()
plt.show()

Each backend induces a qualitatively different similarity structure. Raw-pixel MMD is dominated by low-level statistics (color, contrast); the pretrained encoder injects ImageNet semantics; the per-dataset autoencoder produces a self-supervised latent that highlights intra-dataset structure.

## 4. Random Fourier Features mode — explicit per-dataset vectors

Switching `mode='rff'` returns a single fixed-dim vector μ̂_P per dataset (eq. 3.27 of MMD.pdf). The Euclidean distance between two such vectors approximates `MMD(P, Q)`.

In [ ]:
embedder = MMDEmbedder(
    mode='rff',
    kernel='rbf',
    bandwidth='median',
    encoder=RawEncoder(),
    n_rff=1024,
    max_samples=300,
    device='cpu',
    seed=0,
)
rff_vecs = embedder.embed(raw_datasets).cpu().numpy()
print('RFF embedding shape:', rff_vecs.shape)

rff_D = np.linalg.norm(rff_vecs[:, None, :] - rff_vecs[None, :, :], axis=-1) ** 2
plt.figure(figsize=(5, 4))
sns.heatmap(
    rff_D,
    annot=True,
    fmt='.3f',
    xticklabels=DATASET_NAMES,
    yticklabels=DATASET_NAMES,
    cbar=True,
    cmap='viridis_r',
)
plt.title(r'$\|\hat\mu_P - \hat\mu_Q\|^2$ via Random Fourier Features')
plt.tight_layout()
plt.show()

## 5. One-line `mmd()` convenience function

For the simple two-dataset case the convenience function exposes the entire pipeline in a single call — useful inside notebooks or scripts.

In [ ]:
for i, j in [(0, 1), (0, 2), (2, 3)]:
    val = float(mmd(raw_datasets[i], raw_datasets[j], kernel='rbf', estimator='biased', max_samples=300))
    print(f'MMD²({DATASET_NAMES[i]}, {DATASET_NAMES[j]}) = {val:.4f}')